In [1]:
from pathlib import Path
import importlib
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.disaster_ml as disaster_ml
importlib.reload(disaster_ml)
from src.disaster_ml import (
    evaluate_models,
    load_data,
    prepare_training_data,
    predict_disaster_probabilities,
)

DATA = load_data(
    PROJECT_ROOT / "data" / "INMET",
    PROJECT_ROOT / "data" / "S2ID",
    PROJECT_ROOT / "data" / "IBGE",
)

print("Dados carregados:")
for nome, tabela in DATA.items():
    print(f"{nome}: {tabela.shape}")

print("Valores faltantes nas variaveis meteorologicas apos imputacao:")
weather_columns = list(disaster_ml.WEATHER_COLUMNS.values())
print(DATA["inmet"][weather_columns].isna().sum().to_dict())

TRAINING_DATA = prepare_training_data(
    DATA,
    window_days=1,
    positive_ratio=0.5,
    noise_scale=0.05,
)
print(f"dados de treinamento em janelas de N dias: {TRAINING_DATA.shape}")

Dados carregados:
inmet: (1262592, 21)
s2id: (684, 54)
ibge: (295, 3)
cidade_estacao: (295, 2)
Valores faltantes nas variaveis meteorologicas apos imputacao:
{'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)': 0, 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)': 0, 'UMIDADE RELATIVA DO AR, HORARIA (%)': 0, 'VENTO, VELOCIDADE HORARIA (m/s)': 0, 'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)': 0}
dados de treinamento em janelas de N dias: (78025, 13)


In [2]:
import pandas as pd

from src.disaster_ml import DISASTERS, TARGET_COLUMNS, _normalise_text

print("Distribuicao dos rotulos apos o over sampling:")
positive_rows = TRAINING_DATA[TARGET_COLUMNS].any(axis=1)
label_summary = pd.DataFrame({
    "desastre": DISASTERS,
    "positivos": TRAINING_DATA[TARGET_COLUMNS].sum().astype(int).to_numpy(),
    "total_janelas": len(TRAINING_DATA),
})
label_summary["taxa_positivos"] = label_summary["positivos"] / label_summary["total_janelas"]
print(label_summary.to_string(index=False, formatters={"taxa_positivos": "{:.6%}".format}))
print(f"janelas positivas: {positive_rows.sum()}")
print(f"janelas negativas: {(~positive_rows).sum()}")
print(f"percentual de janelas positivas: {positive_rows.mean():.6%}")

s2id_dates = pd.to_datetime(DATA["s2id"]["Registro"], dayfirst=True, errors="coerce")
ibge_cities = {_normalise_text(city) for city in DATA["ibge"]["cidade"]}
s2id_cities = DATA["s2id"]["Município"].map(_normalise_text)
unmatched_cities = sorted(set(s2id_cities.dropna()) - ibge_cities)

print("\nValidade das relacoes:")
print(f"registros S2ID com data invalida: {s2id_dates.isna().sum()}")
print(f"municipios S2ID sem correspondencia no IBGE: {len(unmatched_cities)}")
if unmatched_cities:
    print("exemplos:", unmatched_cities[:10])
print(f"cidades IBGE sem estacao relacionada: {len(DATA['ibge']) - len(DATA['cidade_estacao'])}")

Distribuicao dos rotulos apos o over sampling:
     desastre  positivos  total_janelas taxa_positivos
   enxurradas      14431          78025     18.495354%
  alagamentos       6328          78025      8.110221%
   inundacoes       3719          78025      4.766421%
deslizamentos       3182          78025      4.078180%
janelas positivas: 26008
janelas negativas: 52017
percentual de janelas positivas: 33.332906%

Validade das relacoes:
registros S2ID com data invalida: 0
municipios S2ID sem correspondencia no IBGE: 0
cidades IBGE sem estacao relacionada: 0


In [3]:
EXEMPLOS_POSITIVOS = TRAINING_DATA.loc[positive_rows].copy()
EXEMPLOS_POSITIVOS["tipos_desastre"] = EXEMPLOS_POSITIVOS[TARGET_COLUMNS].apply(
    lambda linha: ", ".join(
        desastre for desastre, presente in zip(DISASTERS, linha) if presente
    ),
    axis=1,
)

colunas_exemplo = [
    "estacao",
    "data",
    "precip_total",
    "precip_max",
    "temp_mean",
    "humidity_mean",
    "wind_max",
    "pressure_mean",
    "tipos_desastre",
]
print(f"Exemplos positivos disponiveis: {len(EXEMPLOS_POSITIVOS)}")
print(EXEMPLOS_POSITIVOS[colunas_exemplo].head(10).to_string(index=False))

Exemplos positivos disponiveis: 26008
                       estacao       data  precip_total  precip_max  temp_mean  humidity_mean  wind_max  pressure_mean          tipos_desastre
                       INDAIAL 2025-06-27      0.080923    0.219896  15.354135      84.819653  3.972741    1011.306337           deslizamentos
Laguna  - Farol de Santa Marta 2022-12-20     10.963305    1.938402  20.284694      94.329609 13.582085    1010.260511           deslizamentos
                     URUSSANGA 2025-02-03      0.464096    1.119179  24.982739      81.671537  3.234908    1003.485074              enxurradas
                        ITAPOA 2024-07-19      0.000000    0.000000  16.372486      89.113919  3.159671    1024.962092             alagamentos
                       INDAIAL 2023-10-04     94.099574   18.353035  18.105823      94.346807  2.547817     997.116453             alagamentos
                     ARARANGUA 2022-06-03      2.991578    1.835351  13.267132      80.185788  3.445139 

In [4]:
MODELS = evaluate_models(TRAINING_DATA)

print("Modelos e metricas no conjunto de teste:")
for nome, resultado in MODELS.items():
    metricas = resultado["metrics"]
    print(f"\n{nome}:")
    for metrica, valor in metricas.items():
        print(f"  {metrica}: {valor:.4f}")

Modelos e metricas no conjunto de teste:

logistic_regression:
  accuracy: 0.2446
  precision_macro: 0.1576
  recall_macro: 0.7010
  f1_macro: 0.2446

random_forest:
  accuracy: 0.9733
  precision_macro: 0.9657
  recall_macro: 0.9434
  f1_macro: 0.9543

hist_gradient_boosting:
  accuracy: 0.9105
  precision_macro: 0.9639
  recall_macro: 0.7937
  f1_macro: 0.8696


In [5]:
from src.disaster_ml import export_best_model

BEST_MODEL = export_best_model(
    TRAINING_DATA,
    MODELS,
    PROJECT_ROOT / "models" / "best_model.joblib",
    city_station=DATA["cidade_estacao"],
)
print(
    f"Modelo exportado: {BEST_MODEL['model_name']} | "
    f"{BEST_MODEL['selection_metric']}: {BEST_MODEL['selection_score']:.4f} | "
    f"arquivo: models/best_model.joblib"
)

Modelo exportado: random_forest | f1_macro: 0.9543 | arquivo: models/best_model.joblib


In [6]:
ENTRADAS = pd.DataFrame([
    {
        "estacao": "URUSSANGA",
        "precip_total": 0.5,
        "precip_max": 1.1,
        "temp_mean": 24,
        "humidity_mean": 80.8,
        "wind_max": 3,
        "pressure_mean": 1000,
    }
])

PROBABILIDADES = predict_disaster_probabilities(MODELS, ENTRADAS)

for nome_modelo, resultado in PROBABILIDADES.items():
    print(f"\nProbabilidades pelo modelo {nome_modelo}:")
    tabela = resultado.copy()
    tabela.index = [f"entrada_{index + 1}" for index in tabela.index]
    print(tabela.to_string(float_format=lambda valor: f"{valor:.4%}"))


Probabilidades pelo modelo logistic_regression:
           enxurradas  alagamentos  inundacoes  deslizamentos
entrada_1    61.9521%     55.3080%    28.9963%       44.9367%

Probabilidades pelo modelo random_forest:
           enxurradas  alagamentos  inundacoes  deslizamentos
entrada_1    72.0000%      5.0000%     0.0000%        6.0000%

Probabilidades pelo modelo hist_gradient_boosting:
           enxurradas  alagamentos  inundacoes  deslizamentos
entrada_1    59.5221%      6.7466%     1.2715%        0.2581%
